# Reading the life of a cloud from model data
### A hands-on companion to *Intro to Cloud Microphysics Modeling*

Welcome! Over this notebook you'll build physical intuition for the ideas in the talk by
**plotting real output from Cloud Model 1** (CM1) for two classic cloud types:

- **DYCOMS** — marine **stratocumulus**: the flat, gray ocean "cloud deck."
- **RICO** — trade-wind **shallow cumulus**: the popcorn fair-weather clouds over warm oceans.

For each cloud type you have **16 simulations that rain very different amounts** — from steady
drizzle down to almost nothing. They differ by a *single* controllable factor. Figuring out what
that factor is, and why it sets the rain rate, is the thread we'll pull all the way to the research
question at the end.

Every **Exercise** comes with a hidden **▶ Answer** so you can check yourself.

**How to use this notebook**
- Run cells top to bottom (`Shift+Enter`).
- At each **Exercise**, reason out what you expect, then expand the hidden ▶ Answer to check.
- One convention worth fixing now (same as the talk): droplet size means **diameter** $D$, and a
  *kth moment* of the size distribution is $M_k = \sum N(D) \cdot D^k$, where N(D) is the numder concentration of droplet with size $D$. We'll unpack that in §1.


## Setup

We use `xarray` (labeled N-dimensional arrays, basically arrays that remembers what each axis means), `numpy`, and `matplotlib`. Just two small helpers in this cell; everything else you'll read plainly.

In [ ]:
%matplotlib inline
import os, re
import numpy as np
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import fsspec
import glob

plt.rcParams.update({"figure.figsize": (7, 4.2), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 11})


# DATA_DIR = "/data1/shared/MF_REU_data/"
# def list_cases(case):
#     "The 16 simulation IDs for a case, ordered. (What makes them differ is the puzzle of section 4.)"
#     f = glob.glob(f"{DATA_DIR}/{case}/{case.lower()}_na*_r1.nc")
#     ids = [re.search(r"_na([\d.]+)_", os.path.basename(p)).group(1) for p in f]
#     return sorted(ids, key=float)

DATA_DIR = "gs://leap-persistent/arthurzqhu/model_output"
def list_cases(case):
    "The 16 simulation IDs for a case, ordered. (What makes them differ is the puzzle of section 4.)"
    fs = fsspec.filesystem('gs')
    f = fs.glob(f"{DATA_DIR.rstrip('/')}/{case}/{case.lower()}_na*_r1.nc")
    ids = [re.search(r"_na([\d.]+)_", os.path.basename(p)).group(1) for p in f]
    
    return sorted(ids, key=float)

def open_case(case, cid):
    "Open one simulation. case in {'DYCOMS','RICO'}; cid is an ID from list_cases()."
    p = f"{DATA_DIR}/{case}/{case.lower()}_na{cid}_r1.nc"
    return xr.open_dataset(p, engine='h5netcdf', decode_timedelta=False)   # keep time as plain seconds

print("Cloud types: DYCOMS (stratocumulus),  RICO (shallow cumulus)")
print("Simulations per cloud type:", len(list_cases("DYCOMS")))


## §0 · Orientation — what *is* this dataset?

Think of one simulation as a stack of **MRI scans of a cloud**. The cloud lives in a
box of air; we slice the box into a 3-D grid and, at every little cube (a "grid cell" or "grid box"), we record
temperature, wind, water vapor, and how many droplets there are. Each `time` is one fresh scan.

Let's open one **DYCOMS stratocumulus** simulation and look inside the box.

In [ ]:
ds = open_case("DYCOMS", "10")
print(ds.dims)
print("\nSome variables you already have intuition for, recorded at every grid cell:")
for v in ["Qc", "Qr", "temperature_K", "RH", "w"]:
    print(f"  {v:14s} {ds[v].long_name}  [{ds[v].units}]")


Now let's measure the box itself — its size, its grid spacing, and how long the run lasts.

In [ ]:
x, y, z, t = ds.x.values, ds.y.values, ds.z.values, ds.time.values
print(f"Domain width  (x): {x.max()-x.min():.2f} km across {len(x)} points "
      f"-> spacing {(x[1]-x[0])*1000:.0f} m")
print(f"Domain depth  (z): {z.min():.2f} to {z.max():.2f} km across {len(z)} levels")
print(f"Vertical spacing near cloud: {(z[40]-z[39])*1000:.0f} m")
print(f"Run length: {(t[-1]-t[0])/3600:.1f} hours, in {len(t)} snapshots "
      f"({(t[1]-t[0])/60:.0f} min apart)")


**Exercise.** A grid cell is about **100 m** across, but a cloud droplet is about **10 µm**
(`0.00001 m`). Roughly how many factors of ten separate the two? Given that gap, can the model
realistically track where *every individual droplet* sits — and if not, what could it track instead?

<details><summary>▶ Answer</summary>

About **seven orders of magnitude** (100 m vs `1e-5` m). A single 100 m cell holds *billions* of
droplets, so following them one by one is hopeless. Instead the model tracks the **distribution of
droplet sizes** inside each cell — how many large vs small drops there are. Getting an intuition for
that distribution is the whole job of §1.

*(For the record: the box is ~6.3 km wide, ~1.5 km tall, with ~100 m horizontal cells, and the run
covers ~6 hours.)*
</details>

## §1 · Droplet size distributions & moments

A cell holds billions of droplets of many sizes. We summarize them with a **droplet size
distribution (DSD)**, `N(D)` = how many droplets there are at each diameter `D`. The model is a
**bulk scheme**: it doesn't store the whole curve — it stores a few **moments**

$$M_k = \sum N(D)\,D^k$$

It does **not** store the whole curve — only a handful of those summary numbers. `M0` = total
number of droplets, `M3` ∝ total water (mass ∝ volume ∝ `D³`), and `M6` ≈ what a weather radar sees.
We never have to assume a particular curve shape: the moments themselves are what the model carries
and what we'll reason with.

### §1a · Why higher moments "see" the big droplets

Imagine a fruit bowl: **a thousand blueberries** and **one melon**.
- *How many fruit?* (`M0`) → the blueberries win, easily.
- *Total volume of fruit?* (`~M3`) → the melon alone rivals all the berries.
- *"M6"* (sixth power of size) → the melon utterly dominates; the berries vanish.

Higher moments = raising size to a higher power = **more dominated by big droplets**. Let's see it with a toy bag of
just four droplet sizes.

In [ ]:
# A toy "bag" of droplets: four sizes (diameters in um) and how many of each.
D_um   = np.array([10.,   20.,   40.,   80.])     # diameters
counts = np.array([1000., 100.,  10.,   1.])      # one big drop per ~1000 tiny ones

shares = {}
for k in [0, 3, 6]:
    contrib = counts * D_um**k        # each size's contribution to moment M_k
    shares[k] = contrib / contrib.sum()
    print(f"M{k}: share from each size [10, 20, 40, 80 um] = {np.round(shares[k], 3)}")

xb = np.arange(4)
plt.figure()
for off, k, c in [(-0.25, 0, "C0"), (0.0, 3, "C1"), (0.25, 6, "C2")]:
    plt.bar(xb + off, shares[k], 0.25, color=c, label=f"M{k}")
plt.xticks(xb, ["10 um", "20 um", "40 um", "80 um"])
plt.ylabel("share of the moment"); plt.legend(title="moment", loc='upper center')
plt.title("Which droplets does each moment 'see'?   higher k -> bigger drops")
plt.show()


**Exercise.** That single biggest droplet (80 µm) is just **1 in 1111** of all droplets in the example above.
What *fraction of M6* does that one droplet contribute? Work it out by hand, then run the cell to check.

In [ ]:
big = counts[-1] * D_um[-1]**6
allM6 = (counts * D_um**6).sum()
print(f"The one 80um drop is {counts[-1]/counts.sum()*100:.2f}% of the droplets, "
      f"but {big/allM6*100:.0f}% of M6.")

<details><summary>▶ Answer</summary>

About **84%**. One droplet in a thousand carries the *vast majority* of `M6`. This is the key
intuition: **`M0` is a headcount of the many small drops; `M6` is a spotlight on the few giant
drops** (the ones that fall out as rain). Different moments tell us about different parts of
the population.
</details>

### §1b · The width problem — *the heart of the project*

As mentioned in the intro talk, a size distribution has **three independent knobs you can tune**: *how much* water there is, the *typical size* of the drops, and the **width** of the spread (are the drops all nearly the same size, or a mixture of tiny and large?). But to stay cheap, a bulk scheme usually carries only **two** numbers per cell: the **number** of drops (`M0`) and the **mass** of water (`M3`).

**Three things to pin down, only two constraints.** Something is left undetermined — and it's the **width**. To see why that matters, here are two "bags" of droplets built to have the *exact same* number and the *exact same* mass, but very different widths.

In [ ]:
def moments(D, N):
    "Moments M0, M3, M6 of a handful of droplet sizes D (um) with counts N."
    return {k: float(np.sum(N * D**k)) for k in (0, 3, 6)}

# Bag A -- "narrow": every droplet is the same size.
DA, NA = np.array([20.]), np.array([1000.])
# Bag B -- "drizzle tail": 990 small drops + 10 big (60 um) drops, with the small size
# chosen so the TOTAL number and TOTAL mass exactly match Bag A.
Db, Nb = 60., 10.
Ds = ((moments(DA, NA)[3] - Nb*Db**3) / (1000 - Nb))**(1/3)
DB, NB = np.array([Ds, Db]), np.array([1000 - Nb, Nb])

A, B = moments(DA, NA), moments(DB, NB)
print(f"Bag A (narrow):       M0={A[0]:.0f}, M3={A[3]:.2e}, M6={A[6]:.2e}")
print(f"Bag B (drizzle tail): M0={B[0]:.0f}, M3={B[3]:.2e}, M6={B[6]:.2e}")
print(f"Bag B has just {Nb:.0f} extra big drops, but its M6 is {B[6]/A[6]:.1f}x larger.")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(DA, NA, width=2.5, color="C0", label="Bag A: narrow")
ax[0].bar(DB, NB, width=2.5, color="C3", alpha=0.75, label="Bag B: + drizzle tail")
ax[0].set(xlabel="droplet diameter [um]", ylabel="number of drops", yscale="log",
          title="Two bags of droplets"); ax[0].legend()
ax[1].bar(["M0\n(number)", "M3\n(mass)", "M6\n(radar)"],
          [B[0]/A[0], B[3]/A[3], B[6]/A[6]], color=["C7", "C7", "C3"])
ax[1].axhline(1, ls="--", c="gray")
ax[1].set(ylabel="Bag B / Bag A", title="Identical number & mass -- but not M6")
plt.tight_layout(); plt.show()


**Exercise.** Bags A and B agree on `M0` and `M3` to the last digit. If you only knew those
two numbers, could you tell the bags apart? Which one would a weather radar (which essentially sees
`M6`) call "raining" — and what physically makes the difference?

<details><summary>▶ Answer</summary>

No — `M0` and `M3` are **identical**, so by number and mass alone the two bags are indistinguishable.
Yet Bag B's `M6` is **~8× larger**, entirely because of the **10 big drops in its tail**. A radar
would flag Bag B, not Bag A. The thing that differs is the **width** of the distribution, and the
large-drop tail it creates is exactly where rain comes from. `M0` and `M3` are blind to that tail;
`M6` shines a light on it.
</details>

Is this just a toy, or does the same ambiguity live in the real model output?
Let's take every cloudy cell, keep only those that share **nearly the same `M0` and `M3`**, and ask
two things: how much does their width still vary, and does the width have anything to do with rain?

First, how do we put a single *number* on "width"? We can't just read off `M6`: a large `M6` might
mean lots of water, or unusually big drops, or a genuinely wide tail — it blends all three together.
To pull out the **width by itself**, we have to *divide out* the "how much water" and "how big on
average" that we already track in `M0` and `M3`:

$$K_6 = \frac{M_0\,M_6}{M_3^{\,2}}$$

The reason for combining the moments in exactly this way is that the **units cancel** — `K6` is
**dimensionless**. A dimensionless quantity cannot depend on the overall scale of things: multiply
every cell's drop *count* by ten, or rescale all the drop *sizes*, and `K6` doesn't budge. What's
left is purely the *shape* of the distribution. A larger `K6` simply
means a wider distribution with a fatter large-drop tail.

In [ ]:
d = open_case("DYCOMS", "10").isel(time=slice(12, None))   # 2nd half of run (quasi-steady)
M0 = d.M0.values; M3 = d.M3.values; M6 = d.M6.values
Qc = d.Qc.values; Qr = d.Qr.values
cloudy = (Qc > 1e-4) & (M0 > 1e3) & (M3 > 0) & np.isfinite(M6)
M0, M3, M6, Qr = M0[cloudy], M3[cloudy], M6[cloudy], Qr[cloudy]

K6 = M0 * M6 / M3**2 # a pure "width" number

# cells that share nearly the same number AND mass (within ~5%):
same = (np.abs(np.log(M0) - np.median(np.log(M0))) < 0.05) & \
       (np.abs(np.log(M3) - np.median(np.log(M3))) < 0.05)
print(f"{same.sum()} cells share nearly the same M0 and M3,")
print(f"  yet their width K6 still varies {np.percentile(K6[same],90)/np.percentile(K6[same],10):.0f}x.")

lo = K6 < np.percentile(K6, 33); hi = K6 > np.percentile(K6, 67)
print(f"Median rain water:  narrow cells {np.median(Qr[lo])*1e6:.0f} mg/kg"
      f"  vs  wide cells {np.median(Qr[hi])*1e6:.0f} mg/kg")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(K6[same], bins=40, color="C3", alpha=0.8)
ax[0].set(xlabel="K6", ylabel="# of cells",
          title="Cells with the SAME number & mass\nstill span a wide range of widths")
rainy = Qr > 0
ax[1].scatter(K6[rainy][::7], Qr[rainy][::7]*1e6, s=5, alpha=0.25)
ax[1].set(xlabel="width  K6 = M0*M6/M3$^2$",
          ylabel="rain water Qr [mg/kg]", title="Wider distribution -> more rain")
plt.tight_layout(); plt.show()

That's not very helpful. You never want your plots to have all the data points squeezed into one corner and have most of your canvas blank. A neat trick to visualize data that **span across many orders of magnitudes** is to change the axis from linear (default) to **log scale**:

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
bins = np.logspace(np.log10(K6[same].min()), np.log10(K6[same].max()), 40)
ax[0].hist(K6[same], bins=bins, color="C3", alpha=0.8)
ax[0].set(xscale="log", xlabel="K6", ylabel="# of cells",
          title="Cells with the SAME number & mass\nstill span a wide range of widths")
rainy = Qr > 0
ax[1].scatter(K6[rainy][::7], Qr[rainy][::7]*1e6, s=5, alpha=0.25)
ax[1].set(xscale="log", yscale="log", xlabel="width  K6 = M0*M6/M3$^2$",
          ylabel="rain water Qr [mg/kg]", title="Wider distribution -> more rain")
plt.tight_layout(); plt.show()

OK that's a lot more informative.

Two clouds can hold the **identical number of droplets** and the **identical amount of water**, but
one rains and the other doesn't — the difference is the **width** of the size distribution. The data
confirm the toy: among the tens of thousands of cells that share nearly the same `M0` and `M3`, the
width can still vary **>10×**, and the **wider** cells rain several times harder.

So width is a *real, large, rain-controlling* degree of freedom that number and mass alone do not
pin down. The cure is to carry one or two more moments: **`M4` and `M6`** (normalized, as in `K6`)
**are** the width — with `M6` watching the rain-making tail.

**This is the theme of this project.** A bulk scheme cheaply carries number (`M0`) and mass (`M3`) but loses the
width. In the mathematical world, **that information is gone**.
But we live in a physical world bounded by the laws of physics, so perhaps there's some underlying links between everything that would allow us to **recover that width** to some extent.
Concretely, train a model to predict **`M4` and/or `M6`** from `M0, M3` plus the local
*environment* — restoring the width, and with it, realistic rain. 

But what are some important environmental quantities and why?

## §2 · A little thermodynamics — why clouds exist (and the *environment* the ML will use)

Warmer air can hold far more invisible water vapor than cold air — which is why your **breath fogs into a little cloud on a winter day**: warm, moist air from your lungs hits the cold outside, cools, and can no longer hold all its water, so the excess condenses into visible droplets (in liquid form). A real cloud form similarly, it just cools itself a different way: 
1. lift a parcel of moist air to a higher altitude. 
2. Air pressure drops as you go higher because there's less weight exerted by all the air from above.
3. The parcel of moist air expands due to the lower surrounding pressure.
4. Expansion = work, but doing work requires energy. Where does that energy come from?
5. Its own internal energy. So rising = cooling, and that's why it gets cooler when you go higher in altitude.
6. Cool it enough and its vapor **condenses into the droplets**.

Two important variables defines the vertical structure of the cloud layer:
- **Potential temperature `θ`** — **not a measure of temperature but of buoyancy.** Intuitively, Warmer air (higher θ) → less dense → more buoyant. Defined as temperature a parcel *would* have if you bring it to the ground, so parcels at different heights can be compared fairly. 
  - **Why do we need to compare them?** Because clouds form when (moist) air rises, and air rises when it's buoyant (lighter than its surroundings). If you have more buoyant air (higher θ) below less buoyant air (lower θ), convection will happen and we can expect cloud to form. 
- **Relative humidity `RH`** — how close the air is to its moisture limit. `RH = 100%` means full, and that's where cloud forms.

These (plus pressure and height) are exactly the **environmental predictors** the width model in §1b needs. Let's look at the average vertical profile of the DYCOMS deck.

In [ ]:
ds = open_case("DYCOMS", "10")
prof = ds.isel(time=slice(12, None)).mean(dim=["time", "x", "y"])
z = ds.z.values
theta = prof.temperature_theta.values
qv = prof.mixing_ratio.values * 1e3      # g/kg
rh = prof.RH.values

fig, ax = plt.subplots(1, 3, figsize=(12, 4.6), sharey=True)
ax[0].plot(theta, z, "C0", lw=2); ax[0].set(xlabel="θ [K]", ylabel="height [km]", title="potential temperature")
ax[1].plot(qv, z, "C2", lw=2);    ax[1].set(xlabel="water vapor [g/kg]", title="vapor")
ax[2].plot(rh, z, "C3", lw=2); ax[2].axvline(100, ls="--", c="gray"); ax[2].set(xlabel="RH [%]", title="relative humidity")
for a in ax: a.set_ylim(0, 1.2)
plt.tight_layout(); plt.show()


**Exercise.** (a) What does it mean to have a layer of air with the same θ? (b) What if you have more buoyant air (higher θ) *above* less buoyant air (lower θ)? 

From the figures you generated above, can you find layers of air that roughly corresponds to (a) and (b)? 

<details><summary>▶ Answer</summary>

(a) A layer with the same θ = a neutrally-buoyant layer.
If every parcel in a layer has the same θ, then moving a parcel up or down leaves it matching its new surroundings exactly — it feels no push to keep rising and none to sink back. There's no buoyancy "restoring
force," so air mixes up and down freely. In fact a uniform θ is the fingerprint of a well-mixed layer: turbulence and convection have churned the layer until its buoyancy is homogenized, the way stirring a pot evens out
its temperature. (And because the air is mixing freely, anything it carries — like water vapor — gets blended uniformly too. Hold that thought.)

(b) Higher θ (more buoyant) sitting above lower θ (less buoyant) = a stable layer — a lid.
If the light, buoyant air is already on top and the heavy air is already on the bottom, like oil floating on water. Nothing wants to move. If you try to shove a low-θ parcel up into the high-θ air above, it's now denser than its surroundings and immediately sinks back. So this configuration **suppresses vertical motion** and acts as a cap. (Contrast with the unstable case mentioned previously — buoyant air below heavy air — which overturns into convection.)

Finding them in the plot:
- (a) is the boundary layer below ~790 m, especially below ~500 m, where θ is nearly constant: the well-mixed layer.
- (b) is the sharp θ jump at ~790 m, often referred to as the (temperature) inversion, where θ leaps up over a thin layer — strongly stable — and stays high above it. That jump is the lid.

</details>

## §3 · A little dynamics — what moves the air

A cloud layer is a **lava lamp** (or a pot of water just before the boil): warm blobs
rise, cool blobs sink, in slow overturning loops. The model variable `w` is the **vertical air
velocity** (m/s) — positive = rising. Clouds are not painted on the sky at random; they live where
the air is going **up**. The link is clearest for **cumulus** clouds (each is a buoyant tower), so
we switch to **RICO** here. Let's slice the box vertically and look.

In [ ]:
ds = open_case("RICO", "10")
tt = -1
Qc = ds.Qc.isel(time=tt)
yj = int(Qc.sum(dim=["x", "z"]).argmax())     # the y-slice with the most cloud
x, z = ds.x.values, ds.z.values
w_xz  = ds.w.isel(time=tt, y=yj).values.T     # -> (z, x)
Qc_xz = Qc.isel(y=yj).values.T

plt.figure(figsize=(9, 4.6))
vmax = np.nanpercentile(np.abs(w_xz), 99)
pc = plt.pcolormesh(x, z, w_xz, cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto")
plt.colorbar(pc, label="vertical velocity w [m/s]  (red = up)")
plt.contour(x, z, Qc_xz, levels=[1e-4], colors="k", linewidths=1.2)
plt.ylim(0, 2.5); plt.xlabel("x [km]"); plt.ylabel("height [km]")
plt.title("RICO cumulus: cloud towers (black outline) sit in rising (red) air")
plt.show()


**Exercise.** Is cloud water (`Qc`) preferentially found where `w > 0`? Make the scatter
and decide.

<details><summary>▶ Answer</summary>

```
tt = -1
w_all  = ds.w.isel(time=tt).values.ravel()
Qc_all = ds.Qc.isel(time=tt).values.ravel() * 1e3   # g/kg
s = np.random.default_rng(0).choice(w_all.size, 4000, replace=False)
plt.figure()
plt.scatter(w_all[s], Qc_all[s], s=6, alpha=0.3)
plt.axvline(0, c="gray", ls="--")
plt.xlabel("vertical velocity w [m/s]"); plt.ylabel("cloud water Qc [g/kg]")
plt.title("Cloud water lives in updrafts")
plt.show()
frac = np.mean(w_all[Qc_all > 0.01] > 0)
print(f"Of cloudy points, {frac*100:.0f}% are in rising air (w > 0).")
```

About **two-thirds (~67%)** of cloudy points sit in rising air, and the densest cloud water lines up
with the strongest updrafts: condensation happens in **updrafts**, because rising air is the air
that's expanding, cooling, and wringing out its vapor. This is why `w` is such a useful
*environmental* predictor — it tells you where droplets are actively being made. (Downdrafts do the
opposite: sinking air warms and **evaporates** cloud — which is why a *stirred* stratocumulus deck
like DYCOMS, with cloud in both up- and downdrafts, gives a much weaker ~50/50 split.)
</details>

## §4 · Microphysics — from aerosol seeds to rain

The 16 simulations differ in just one thing: **how much aerosol** the air holds — the tiny particles that cloud droplets must form on.
Here's why that single knob controls the rain:

- Aerosol particles are like **seeds** (formally, cloud condensation nuclei) that
  droplets grow on. Same water, *more seeds* → many **small** droplets; *few seeds* → fewer **big**
  droplets.
- **The raindrop race.** Bigger drops fall faster. Rain happens when drops of *different* sizes fall
  at *different* speeds, so the fast big ones catch and merge with slow small ones
  (**collision–coalescence**). If every drop were identical they'd fall in lockstep — a
  **conveyor belt**, no catching up, no collision, no rain. *Width* is what breaks the lockstep.

Let's contrast the lowest- and highest-aerosol DYCOMS runs.

In [ ]:
lo = open_case("DYCOMS", "10")    # few seeds
hi = open_case("DYCOMS", "120")   # many seeds

def cloud_mean_profile(ds, var):
    "Horizontal+time mean of `var`, counting only cloudy cells."
    d = ds.isel(time=slice(12, None))
    cloudy = d.Qc > 1e-5
    return d[var].where(cloudy).mean(dim=["time", "x", "y"]).values

z = lo.z.values
fig, ax = plt.subplots(1, 3, figsize=(12, 4.6), sharey=True)

# reff is "effective radius". Don't worry about the exact definition of it, just know that it's a measure of size.

ax[0].plot(cloud_mean_profile(lo, "reff")*1e6, z, "C0", lw=2, label="Na=10 (few seeds)")
ax[0].plot(cloud_mean_profile(hi, "reff")*1e6, z, "C3", lw=2, label="Na=120 (many seeds)")
ax[0].set(xlabel="cloud droplet reff [um]", ylabel="height [km]", title="droplet SIZE"); ax[0].legend()
ax[1].plot(cloud_mean_profile(lo, "Nc")/1e6, z, "C0", lw=2)
ax[1].plot(cloud_mean_profile(hi, "Nc")/1e6, z, "C3", lw=2)
ax[1].set(xlabel="cloud number Nc [1/cc]", title="droplet COUNT")
ax[2].plot(lo.Qr.isel(time=slice(12,None)).mean(["time","x","y"]).values*1e6, z, "C0", lw=2)
ax[2].plot(hi.Qr.isel(time=slice(12,None)).mean(["time","x","y"]).values*1e6, z, "C3", lw=2)
ax[2].set(xlabel="rain water Qr [mg/kg]", title="RAIN")
for a in ax: a.set_ylim(0, 1.1)
plt.tight_layout(); plt.show()


Does the Na=120 actually not rain at all? Try switching xscale from linear to log to have a closer look. (Check previous plotting scripts if you don't know how.)

Now the "raindrop race": do bigger droplets really fall faster? The model gives a
mass-weighted fall speed `vTM3`. Let's plot fall speed against droplet size.

In [ ]:
tt = -1
reff = lo.reff.isel(time=tt).values.ravel() * 1e6     # μm
vT   = lo.vTM3.isel(time=tt).values.ravel()           # m/s
ok = np.isfinite(reff) & np.isfinite(vT) & (reff > 0)
s = np.random.default_rng(1).choice(np.flatnonzero(ok), 4000, replace=False)
plt.figure()
plt.scatter(reff[s], vT[s], s=6, alpha=0.3)
plt.xlabel("effective radius [um]"); plt.ylabel("fall speed vTM3 [m/s]")
plt.title("The raindrop race: bigger droplets fall faster")
plt.show()


**Exercise.** Without looking below: which run has **bigger** cloud droplets, and which
**rains more** — the few-seed `Na=10` or the many-seed `Na=120`?

<details><summary>▶ Answer</summary>

**`Na=10` (few seeds)** has the bigger droplets *and* far more rain. With few seeds, the fixed water
is shared among few droplets, so each grows large; large drops spread the fall-speed race wide
enough to collide and make rain. Crank the aerosol to `Na=120` and the same water is split among
*many* tiny droplets — nearly uniform, a "conveyor belt," and rain is throttled. This is the
famous **aerosol effect on clouds**, and you've now watched it happen cell by cell. Notice the
mechanism *runs through the width* from §1b.
</details>

## §5 · The aerosol → rain story across *all* runs, and your project

You've seen two endpoints. Now use the **whole 16-member sweep**, for **both** cloud types, to draw
the full relationship between aerosol, droplet size, and rain.

In [ ]:
# This might take a few minutes to run

def summarize(case, cid):
    ds = open_case(case, cid)
    d = ds.isel(time=slice(12, None))
    cloudy = d.Qc > 1e-4
    reff = float(d.reff.where(cloudy).mean())          # m
    qr   = float(d.Qr.mean()) * 1e6                    # mg/kg
    sfc  = float(d.sedflux_M3.isel(z=0).mean())        # surface precip proxy
    ds.close()
    return reff*1e6, qr, sfc

results = {}
for case in ["DYCOMS", "RICO"]:
    cids = list_cases(case)
    rows = np.array([summarize(case, cid) for cid in cids])
    results[case] = (np.array([float(c) for c in cids]), rows)
    print(f"{case}: done ({len(cids)} simulations)")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
for case, c in [("DYCOMS", "C0"), ("RICO", "C1")]:
    na, rows = results[case]
    ax[0].plot(na, rows[:, 0], "o-", color=c, label=case)
    ax[1].semilogy(na, rows[:, 2], "o-", color=c, label=case)
ax[0].set(xlabel="aerosol number Na [/cc]", ylabel="cloud droplet reff [um]",
          title="More aerosol -> smaller droplets"); ax[0].legend()
ax[1].set(xlabel="aerosol number Na [/cc]", ylabel="surface precip (M3 flux, proxy)",
          title="More aerosol -> less rain"); ax[1].legend()
plt.tight_layout(); plt.show()


**Exercise.** Describe the two trends in one sentence each. Then a subtler one: do
**DYCOMS** and **RICO** respond to aerosol the *same* way when it comes to rain?

<details><summary>▶ Answer</summary>

- **Size:** droplet `reff` falls steadily as aerosol rises (≈27→10 µm for DYCOMS, 34→12 µm for
  RICO) — robust and nearly universal.
- **Rain:** surface precipitation drops as aerosol rises, for both.

The subtlety: **DYCOMS rain *collapses* — by ~5 orders of magnitude** — essentially shutting off at
high aerosol, while **RICO keeps drizzling** (it stays hundreds of times rainier than DYCOMS at the
same high `Na`). The thin stratocumulus deck is delicately balanced and easily switched off; the
deeper cumulus has enough depth and updraft to keep making rain. *Same microphysics, different
cloud regime, different climate consequence* — which is exactly why we simulate more than one case.
</details>

### Closing the loop: does rain track the width across the whole sweep?

In §3 you saw that *cloud* water lives in updrafts (`w` vs `Qc`), and in §1b that *wider* distributions rain more — for a single run. Let's fold in the aerosol story: pool **every rainy cloudy cell from all 16 DYCOMS runs**, scatter the **width** (`K6`) against the **rain mass** (`Qr`), and color each point by its aerosol level.

**Exercise.** Before running: (a) do you expect rain to still track width once you mix together clouds with very different aerosol amounts? (b) Where on the scatter do you expect the *high-aerosol* points to land?

In [ ]:
# Pool every rainy cloudy cell across the whole DYCOMS sweep, tagged by aerosol number.
K6, Qr, Na = [], [], []
for cid in list_cases("DYCOMS"):
    d = open_case("DYCOMS", cid).isel(time=slice(12, None))
    qc = d.Qc.values; qr = d.Qr.values
    M0 = d.M0.values; M3 = d.M3.values; M6 = d.M6.values
    m = (qc > 1e-4) & (M0 > 1e3) & (M3 > 0) & np.isfinite(M6) & (qr > 0)
    K6.append(M0[m]*M6[m]/M3[m]**2); Qr.append(qr[m]); Na.append(np.full(m.sum(), float(cid)))
    d.close()
K6 = np.concatenate(K6); Qr = np.concatenate(Qr); Na = np.concatenate(Na)

s = np.random.default_rng(0).choice(K6.size, 6000, replace=False)
plt.figure()
sc = plt.scatter(K6[s], Qr[s]*1e6, c=Na[s], s=6, alpha=0.4, cmap="viridis", norm=matplotlib.colors.LogNorm())
plt.colorbar(sc, label="log10(aerosol number Na)")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("width  K6 = M0*M6/M3$^2$"); plt.ylabel("rain water Qr [mg/kg]")
plt.title("Rain lives in wide distributions")
plt.show()

wide = K6 > np.median(K6)
print(f"The wider half of cloudy cells holds {Qr[wide].sum()/Qr.sum()*100:.0f}% of all the rain water.")

<details><summary>▶ Answer</summary>

Yes. Even after pooling clouds from across the *entire* aerosol sweep, rain still climbs steeply with width — the §1b relationship is universal, not a fluke of one run. In fact the **wider half of cloudy cells holds ~99% of all the rain water**.

And the colors tell the capstone's punchline: **high-aerosol points (yellow) pile up at small width with almost no rain, while low-aerosol points (purple) reach the wide, rainy end.** So the whole chain is

`aerosol → droplet-distribution width → rain`

— aerosol sets the rain by sliding each cloud along this single width–rain curve. That's the deepest reason the project targets the width-carrying moments `M4` and `M6`: pin down the width, and you've pinned down the rain.
</details>

## Where this goes next — your summer project

Tie the threads together:

1. **§1b** — a bulk scheme advects **number (`M0`)** and **mass (`M3`)**, but the **width** of the
   droplet size distribution is a third degree of freedom it can't carry. We showed the missing width is
   *real and large* — and that **`M4` and `M6` encode it** (`M6` watches the rain-making tail).
2. **§2–§3** — the width isn't random: it's shaped by the **environment** (`θ`, vapor, pressure,
   height `z`, vertical velocity `w`) — all variables sitting in these same files.
3. **§4–§5** — width controls the **raindrop race**, and so controls **rain** and the cloud's effect
   on climate.

**The project.** In practice the scheme splits its number and mass into two categories — **cloud** and **rain** — and carries four numbers per grid cell:
- `Nc`, `Nr` — the *number* of cloud droplets and of rain drops (per kg of air). Together they make the total number: `M0 = Nc + Nr`.
- `Qc`, `Qr` — the *mass* of cloud water and of rain water (per kg of air). Together they make the total water: `M3 ∝ Qc + Qr`.

Those four, plus the environment, are the **inputs**; the width-carrying moments `M4` and `M6` are the **outputs** to predict:

$$(\,\underbrace{N_c,\ N_r,\ Q_c,\ Q_r}_{\text{number and mass, by category}},\ \underbrace{T,\ P,\ z,\ w,\ \ldots}_{\text{environment}}\,)\ \ \longrightarrow\ \ \underbrace{(M_4,\ M_6)}_{\text{the lost width}}$$

A cheap bulk scheme thereby regains the width of the droplet size distribution — and predicts rain — like an expensive bin scheme. Every quantity on the left is something you just plotted. You already have the data (`gs://leap-persistent/arthurzqhu/model_output/`) and, now, the intuition.

**Suggested next steps to play with:**
- Build the training table at every cloudy cell, across all 16 runs and both cases. The **input features** are `Nc, Nr, Qc, Qr` together with the environment (`T, P, z, w, …`); the **prediction targets (the ML outputs)** are the moments `M4` and `M6` — keep these two groups clearly separate.
- Plot `M4`/`M6` (normalized, as in §1b) against `w`, `RH`, and height — *which environment
  variables actually predict the width?*
- Within each bin of the input features (`Nc, Nr, Qc, Qr`, and other environment quantities), how large is the spread of M4 and M6? This tells us the identifiability of the width. In other words, how much can we constrain M4 and M6 if we know the input features.
- Compare DYCOMS vs RICO: does a width model trained on one transfer to the other?